In [9]:
%pip install -q "lightgbm>=4,<5" "wandb>=0.19,<1" "optuna>=4,<5"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import lightgbm as lgb
import numpy as np
import wandb

WANDB_ENTITY = "kende23-n-a"
WANDB_PROJECT = "Walmart-Recruiting---Store-Sales-Forecasting"
VALIDATION_WEEKS = 32
HOLIDAY_WEIGHT = 5
SEED = 42

wandb.login()

df_train = pd.read_csv("/content/drive/My Drive/walmart_competition_data/train.csv", parse_dates=["Date"])
df_test = pd.read_csv("/content/drive/My Drive/walmart_competition_data/test.csv", parse_dates=["Date"])
df_features = pd.read_csv("/content/drive/My Drive/walmart_competition_data/features.csv", parse_dates=["Date"])
df_stores = pd.read_csv("/content/drive/My Drive/walmart_competition_data/stores.csv")

df_train_merged = df_train.merge(df_stores, on="Store", how="left")
df_train_merged = df_train_merged.merge(
    df_features,
    on=["Store", "Date", "IsHoliday"],
    how="left",
)

# Time-based validation split: sort chronologically and use the last 32 weekly dates as validation.
df_train_merged = df_train_merged.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)
validation_dates = np.sort(df_train_merged["Date"].unique())[-VALIDATION_WEEKS:]

train_df_split = df_train_merged.loc[~df_train_merged["Date"].isin(validation_dates)].copy()
val_df_split = df_train_merged.loc[df_train_merged["Date"].isin(validation_dates)].copy()

train_df_split = train_df_split.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)
val_df_split = val_df_split.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)

y_train = train_df_split["Weekly_Sales"]
is_holiday_train = train_df_split["IsHoliday"]
X_train = train_df_split.copy()

y_val = val_df_split["Weekly_Sales"]
is_holiday_val = val_df_split["IsHoliday"]
X_val = val_df_split.copy()

split_summary = {
    "validation_weeks": VALIDATION_WEEKS,
    "train_rows": len(X_train),
    "validation_rows": len(X_val),
    "train_start": str(X_train["Date"].min().date()),
    "train_end": str(X_train["Date"].max().date()),
    "validation_start": str(X_val["Date"].min().date()),
    "validation_end": str(X_val["Date"].max().date()),
    "validation_unique_weeks": int(X_val["Date"].nunique()),
}

print(f"Train dates: {X_train['Date'].min().date()} to {X_train['Date'].max().date()}")
print(f"Validation dates: {X_val['Date'].min().date()} to {X_val['Date'].max().date()}")
print(f"Validation unique weeks: {X_val['Date'].nunique()}")
print(f"Train shape: {X_train.shape}")
print(f"Validation shape: {X_val.shape}")


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: nmetr23 (kende23-n-a) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Train dates: 2010-02-05 to 2012-03-16
Validation dates: 2012-03-23 to 2012-10-26
Validation unique weeks: 32
Train shape: (326856, 16)
Validation shape: (94714, 16)


In [10]:
X_train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106
1,1,2,2010-02-05,50605.27,False,A,151315,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106
2,1,3,2010-02-05,13740.12,False,A,151315,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106
3,1,4,2010-02-05,39954.04,False,A,151315,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106
4,1,5,2010-02-05,32229.38,False,A,151315,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106


In [11]:
from __future__ import annotations

from typing import Iterable

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline


MARKDOWN_COLS = ("MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5")
NUMERIC_EXTERNAL_COLS = ("CPI", "Unemployment", "Temperature", "Fuel_Price")


def _existing_columns(frame: pd.DataFrame, columns: Iterable[str]) -> list[str]:
    return [col for col in columns if col in frame.columns]


class WalmartFeatureCleaner(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        date_col: str = "Date",
        markdown_cols: tuple[str, ...] = MARKDOWN_COLS,
        numeric_impute_cols: tuple[str, ...] = NUMERIC_EXTERNAL_COLS,
        add_markdown_missing_indicators: bool = True,
        markdown_fill_value: float = 0.0,
        numeric_impute_strategy: str = "median",
        category_cols: tuple[str, ...] = ("Store", "Dept", "Type"),
    ):
        self.date_col = date_col
        self.markdown_cols = markdown_cols
        self.numeric_impute_cols = numeric_impute_cols
        self.add_markdown_missing_indicators = add_markdown_missing_indicators
        self.markdown_fill_value = markdown_fill_value
        self.numeric_impute_strategy = numeric_impute_strategy
        self.category_cols = category_cols

    def fit(self, X: pd.DataFrame, y=None):
        if self.numeric_impute_strategy not in {"median", "mean", "none"}:
            raise ValueError("numeric_impute_strategy must be 'median', 'mean', or 'none'.")

        self.numeric_fill_values_ = {}
        numeric_cols = _existing_columns(X, self.numeric_impute_cols)
        if self.numeric_impute_strategy != "none":
            for col in numeric_cols:
                if self.numeric_impute_strategy == "median":
                    self.numeric_fill_values_[col] = X[col].median()
                else:
                    self.numeric_fill_values_[col] = X[col].mean()
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        frame = X.copy()

        if self.date_col in frame.columns:
            frame[self.date_col] = pd.to_datetime(frame[self.date_col])

        for col in _existing_columns(frame, self.markdown_cols):
            if self.add_markdown_missing_indicators:
                frame[f"{col}_missing"] = frame[col].isna().astype("int8")
            frame[col] = frame[col].fillna(self.markdown_fill_value)

        for col, value in getattr(self, "numeric_fill_values_", {}).items():
            if col in frame.columns:
                frame[col] = frame[col].fillna(value)

        for col in _existing_columns(frame, self.category_cols):
            frame[col] = frame[col].astype("category")

        if "IsHoliday" in frame.columns:
            frame["IsHoliday"] = frame["IsHoliday"].astype("int8")

        return frame


class CalendarFeatureTransformer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        date_col: str = "Date",
        start_date: str = "2010-02-05",
        add_cyclical_features: bool = True,
        drop_date: bool = False,
    ):
        self.date_col = date_col
        self.start_date = start_date
        self.add_cyclical_features = add_cyclical_features
        self.drop_date = drop_date

    def fit(self, X: pd.DataFrame, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        frame = X.copy()
        date = pd.to_datetime(frame[self.date_col])
        iso = date.dt.isocalendar()

        frame["Year"] = date.dt.year.astype("int16")
        frame["Month"] = date.dt.month.astype("int8")
        frame["WeekOfYear"] = iso.week.astype("int8")
        frame["Quarter"] = date.dt.quarter.astype("int8")
        frame["DayOfYear"] = date.dt.dayofyear.astype("int16")
        frame["DaysFromStart"] = (date - pd.Timestamp(self.start_date)).dt.days.astype("int16")

        if self.add_cyclical_features:
            frame["WeekSin"] = np.sin(2 * np.pi * frame["WeekOfYear"] / 52.0)
            frame["WeekCos"] = np.cos(2 * np.pi * frame["WeekOfYear"] / 52.0)
            frame["MonthSin"] = np.sin(2 * np.pi * frame["Month"] / 12.0)
            frame["MonthCos"] = np.cos(2 * np.pi * frame["Month"] / 12.0)

        if self.drop_date:
            frame = frame.drop(columns=[self.date_col])

        return frame


class WalmartHolidayFeatureTransformer(BaseEstimator, TransformerMixin):

    HOLIDAY_DATES = {
        "SuperBowl": ("2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"),
        "LaborDay": ("2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"),
        "Thanksgiving": ("2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"),
        "Christmas": ("2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"),
    }

    def __init__(
        self,
        date_col: str = "Date",
        add_holiday_flags: bool = True,
        add_proximity_features: bool = True,
    ):
        self.date_col = date_col
        self.add_holiday_flags = add_holiday_flags
        self.add_proximity_features = add_proximity_features

    def fit(self, X: pd.DataFrame, y=None):
        self.holiday_dates_ = {
            name: pd.to_datetime(list(dates)) for name, dates in self.HOLIDAY_DATES.items()
        }
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        frame = X.copy()
        date = pd.to_datetime(frame[self.date_col])

        for name, holiday_dates in self.holiday_dates_.items():
            if self.add_holiday_flags:
                frame[f"Is{name}Week"] = date.isin(holiday_dates).astype("int8")

            if self.add_proximity_features:
                distances = np.vstack([(date - holiday).dt.days.to_numpy() for holiday in holiday_dates])
                nearest_distance = distances[np.abs(distances).argmin(axis=0), np.arange(len(date))]
                frame[f"DaysToNearest{name}"] = np.abs(nearest_distance).astype("int16")
                frame[f"WeeksToNearest{name}"] = (np.abs(nearest_distance) / 7.0).astype("float32")

        return frame


class MarkdownFeatureTransformer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        markdown_cols: tuple[str, ...] = MARKDOWN_COLS,
        add_total_markdown: bool = True,
        add_has_markdown: bool = True,
        add_log_markdowns: bool = True,
        add_holiday_interaction: bool = True,
        holiday_col: str = "IsHoliday",
    ):
        self.markdown_cols = markdown_cols
        self.add_total_markdown = add_total_markdown
        self.add_has_markdown = add_has_markdown
        self.add_log_markdowns = add_log_markdowns
        self.add_holiday_interaction = add_holiday_interaction
        self.holiday_col = holiday_col

    def fit(self, X: pd.DataFrame, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        frame = X.copy()
        markdown_cols = _existing_columns(frame, self.markdown_cols)

        if self.add_total_markdown and markdown_cols:
            frame["TotalMarkDown"] = frame[markdown_cols].sum(axis=1)

        if self.add_has_markdown:
            for col in markdown_cols:
                frame[f"Has{col}"] = (frame[col] > 0).astype("int8")
            if "TotalMarkDown" in frame.columns:
                frame["HasAnyMarkDown"] = (frame["TotalMarkDown"] > 0).astype("int8")

        if self.add_log_markdowns:
            for col in markdown_cols:
                frame[f"{col}_log1p"] = np.log1p(frame[col].clip(lower=0))
            if "TotalMarkDown" in frame.columns:
                frame["TotalMarkDown_log1p"] = np.log1p(frame["TotalMarkDown"].clip(lower=0))

        if self.add_holiday_interaction and self.holiday_col in frame.columns:
            if "TotalMarkDown" in frame.columns:
                frame["Holiday_TotalMarkDown"] = frame[self.holiday_col] * frame["TotalMarkDown"]
            for col in markdown_cols:
                frame[f"Holiday_{col}"] = frame[self.holiday_col] * frame[col]

        return frame


class InteractionFeatureTransformer(BaseEstimator, TransformerMixin):

    def fit(self, X: pd.DataFrame, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        frame = X.copy()
        if {"Store", "Dept"}.issubset(frame.columns):
            frame["Store_Dept"] = (
                frame["Store"].astype("int32") * 1000 + frame["Dept"].astype("int32")
            ).astype("int32")
        if {"Type", "Dept"}.issubset(frame.columns):
            type_codes = frame["Type"].astype("category").cat.codes.astype("int32")
            frame["Type_Dept"] = (type_codes * 1000 + frame["Dept"].astype("int32")).astype("int32")
        return frame



class HistoricalAggregateTransformer(BaseEstimator, TransformerMixin):
    """Time-safe shifted aggregate encodings, aligned with the XGBoost experiment."""

    def __init__(
        self,
        groupings: tuple[tuple[str, ...], ...] = (("Store",), ("Dept",), ("Store", "Dept"), ("Type", "Dept")),
        target_col: str = "Weekly_Sales",
    ):
        self.groupings = groupings
        self.target_col = target_col

    @staticmethod
    def _prefix(cols: tuple[str, ...]) -> str:
        return "_".join(cols) + "_Sales"

    def fit(self, X: pd.DataFrame, y=None):
        frame = X.copy()
        if self.target_col not in frame.columns:
            if y is None:
                raise ValueError(f"{self.target_col!r} must be present or y must be provided.")
            frame[self.target_col] = np.asarray(y)

        self.global_stats_ = frame[self.target_col].agg(["mean", "median", "std"]).to_dict()
        self.maps_ = {}
        for cols in self.groupings:
            existing_cols = tuple(col for col in cols if col in frame.columns)
            if len(existing_cols) != len(cols):
                continue
            self.maps_[existing_cols] = frame.groupby(list(existing_cols), observed=True)[self.target_col].agg(
                ["mean", "median", "std", "count"]
            )
        return self

    def fit_transform(self, X: pd.DataFrame, y=None, **fit_params) -> pd.DataFrame:
        frame = X.copy()
        added_target = False
        if self.target_col not in frame.columns:
            if y is None:
                raise ValueError(f"{self.target_col!r} must be present or y must be provided.")
            frame[self.target_col] = np.asarray(y)
            added_target = True

        frame["__original_order"] = np.arange(len(frame))
        sort_cols = [col for col in ["Date", "Store", "Dept"] if col in frame.columns]
        if sort_cols:
            frame = frame.sort_values(sort_cols).copy()

        for cols in self.groupings:
            existing_cols = tuple(col for col in cols if col in frame.columns)
            if len(existing_cols) != len(cols):
                continue
            grouped = frame.groupby(list(existing_cols), observed=True, sort=False)[self.target_col]
            prefix = self._prefix(existing_cols)
            frame[f"{prefix}_mean"] = grouped.transform(
                lambda s: s.shift(1).expanding(min_periods=1).mean()
            ).astype("float32")
            frame[f"{prefix}_median"] = grouped.transform(
                lambda s: s.shift(1).expanding(min_periods=1).median()
            ).astype("float32")
            frame[f"{prefix}_std"] = grouped.transform(
                lambda s: s.shift(1).expanding(min_periods=2).std()
            ).astype("float32")
            frame[f"{prefix}_count"] = frame.groupby(
                list(existing_cols), observed=True, sort=False
            ).cumcount().astype("int32")

        frame = frame.sort_values("__original_order").drop(columns="__original_order").reset_index(drop=True)
        if added_target:
            frame = frame.drop(columns=[self.target_col])
        self.fit(X, y)
        return frame

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        frame = X.copy().reset_index(drop=True)
        for cols, mapping in self.maps_.items():
            prefix = self._prefix(cols)
            keys = frame[list(cols)] if len(cols) > 1 else frame[cols[0]]
            for stat in ["mean", "median", "std", "count"]:
                values = mapping[stat]
                if len(cols) > 1:
                    lookup_keys = pd.MultiIndex.from_frame(keys)
                    encoded = values.reindex(lookup_keys).to_numpy()
                else:
                    encoded = keys.map(values).to_numpy()
                fallback = self.global_stats_.get(stat, 0.0) if stat != "count" else 0
                frame[f"{prefix}_{stat}"] = pd.Series(encoded).fillna(fallback).to_numpy()
        return frame


def add_safe_lag_52(
    frame: pd.DataFrame,
    observed_history: pd.DataFrame,
    group_cols: tuple[str, ...] = ("Store", "Dept"),
    date_col: str = "Date",
    target_col: str = "Weekly_Sales",
) -> pd.DataFrame:
    """Add same-week-last-year sales using only observed history.

    This is safe for validation/test because the joined target value comes from
    date - 52 weeks in the observed training history, never from the row being
    predicted.
    """
    result = frame.copy()
    history = observed_history[list(group_cols) + [date_col, target_col]].copy()
    history[date_col] = pd.to_datetime(history[date_col]) + pd.DateOffset(weeks=52)
    history = history.rename(columns={target_col: "SalesLag52"})
    history = history.drop_duplicates(list(group_cols) + [date_col], keep="last")

    result = result.merge(history, on=list(group_cols) + [date_col], how="left")
    result["SalesLag52_available"] = result["SalesLag52"].notna().astype("int8")
    result["SalesLag52"] = result["SalesLag52"].astype("float32")
    return result


class ColumnDropper(BaseEstimator, TransformerMixin):

    def __init__(self, columns: tuple[str, ...] = ("Date", "Weekly_Sales"), errors: str = "ignore"):
        self.columns = columns
        self.errors = errors

    def fit(self, X: pd.DataFrame, y=None):
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return X.drop(columns=list(self.columns), errors=self.errors)


class FeatureImportanceSelector(BaseEstimator, TransformerMixin):

    def __init__(self, estimator, threshold: float = 0.0, fit_params: dict | None = None):
        self.estimator = estimator
        self.threshold = threshold
        self.fit_params = fit_params

    def fit(self, X: pd.DataFrame, y):
        fit_params = self.fit_params or {}
        self.estimator.fit(X, y, **fit_params)
        importances = getattr(self.estimator, "feature_importances_", None)
        if importances is None:
            raise ValueError("estimator must expose feature_importances_ after fit.")

        self.feature_importances_ = pd.Series(importances, index=X.columns).sort_values(ascending=False)
        self.selected_features_ = self.feature_importances_[
            self.feature_importances_ > self.threshold
        ].index.tolist()
        if not self.selected_features_:
            raise ValueError("No features passed the importance threshold.")
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return X[self.selected_features_].copy()


def make_walmart_lgbm_feature_pipeline(
    drop_target_and_date: bool = True,
) -> Pipeline:

    pre_processing = [
        ("clean", WalmartFeatureCleaner()),
        ("calendar", CalendarFeatureTransformer()),
        ("holiday", WalmartHolidayFeatureTransformer()),
        ("markdown", MarkdownFeatureTransformer()),
        ("interactions", InteractionFeatureTransformer()),
        ("aggregates", HistoricalAggregateTransformer()),
    ]

    if drop_target_and_date:
        pre_processing.append(("drop_columns", ColumnDropper()))

    return Pipeline(pre_processing)





In [12]:
import optuna
import wandb
import lightgbm as lgb
import matplotlib.pyplot as plt
from wandb.integration.lightgbm import log_summary, wandb_callback

X_train_safe = add_safe_lag_52(X_train, observed_history=X_train)
X_val_safe = add_safe_lag_52(X_val, observed_history=X_train)

feature_pipeline = make_walmart_lgbm_feature_pipeline(
    drop_target_and_date=True
)

feature_engineering_run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    job_type="feature_engineering",
    name="LightGBM_Feature_Engineering",
    tags=["lightgbm", "feature-engineering", "time-split"],
    config={
        **split_summary,
        "safe_lag_features": ["SalesLag52", "SalesLag52_available"],
        "lag52_missing_handling": "left_as_nan_for_lightgbm_native_missing_value_handling",
        "xgboost_aligned_features": ["numeric_Store_Dept", "numeric_Type_Dept", "shifted_target_aggregate_count"],
        "removed_unsafe_features": ["lag_1", "lag_4", "lag_13", "rolling_mean_4", "rolling_std_4", "rolling_mean_13", "rolling_std_13"],
        "drop_target_and_date": True,
        "pipeline_steps": [name for name, _ in feature_pipeline.steps],
    },
    reinit=True,
)

X_train_transformed = feature_pipeline.fit_transform(X_train_safe, y_train)
X_val_transformed = feature_pipeline.transform(X_val_safe)

feature_metadata = pd.DataFrame({
    "feature": X_train_transformed.columns,
    "dtype": X_train_transformed.dtypes.astype(str).values,
    "train_missing_count": X_train_transformed.isna().sum().values,
    "validation_missing_count": X_val_transformed.isna().sum().reindex(X_train_transformed.columns).values,
})

wandb.log({
    "feature_engineering/train_rows": X_train_transformed.shape[0],
    "feature_engineering/validation_rows": X_val_transformed.shape[0],
    "feature_engineering/feature_count": X_train_transformed.shape[1],
    "feature_engineering/categorical_feature_count": int((X_train_transformed.dtypes == "category").sum()),
    "feature_engineering/train_missing_values": int(X_train_transformed.isna().sum().sum()),
    "feature_engineering/validation_missing_values": int(X_val_transformed.isna().sum().sum()),
    "feature_engineering/features": wandb.Table(dataframe=feature_metadata),
})
feature_engineering_run.summary["feature_count"] = X_train_transformed.shape[1]
feature_engineering_run.finish()



/usr/local/lib/python3.12/dist-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


feature_engineering/categorical_feature_count,▁
feature_engineering/feature_count,▁
feature_engineering/train_missing_values,▁
feature_engineering/train_rows,▁
feature_engineering/validation_missing_values,▁
feature_engineering/validation_rows,▁
feature_count,82
feature_engineering/categorical_feature_count,5
feature_engineering/feature_count,82
feature_engineering/train_missing_values,236265
feature_engineering/train_rows,326856


In [14]:
sample_weights_train = np.where(is_holiday_train, HOLIDAY_WEIGHT, 1)
sample_weights_val = np.where(is_holiday_val, HOLIDAY_WEIGHT, 1)

categorical_features = X_train_transformed.select_dtypes(include="category").columns.tolist()

base_params = {
    "objective": "mae",
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1,
    "n_estimators": 100, # Fixed number of estimators
}

feature_selection_run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    job_type="feature_selection",
    name="LightGBM_Feature_Selection",
    tags=["lightgbm", "feature-selection", "importance", "time-split"],
    config={
        **split_summary,
        "holiday_weight": HOLIDAY_WEIGHT,
        "input_feature_count": X_train_transformed.shape[1],
        "categorical_features": categorical_features,
        "selection_rule": "feature_importance > 0",
        **base_params,
    },
    reinit=True,
)

feature_selector_model = lgb.LGBMRegressor(**base_params)
feature_selector_model.fit(
    X_train_transformed,
    y_train,
    sample_weight=sample_weights_train,
    eval_set=[
        (X_train_transformed, y_train),
        (X_val_transformed, y_val),
    ],
    eval_names=["train", "validation"],
    eval_sample_weight=[sample_weights_train, sample_weights_val],
    eval_metric="mae",
    categorical_feature=categorical_features,
    callbacks=[wandb_callback(), lgb.log_evaluation(period=25)],
)
log_summary(feature_selector_model.booster_, save_model_checkpoint=False)

feature_selection_importance = (
    pd.DataFrame({
        "feature": X_train_transformed.columns,
        "importance": feature_selector_model.feature_importances_,
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

selected_features = feature_selection_importance.loc[
    feature_selection_importance["importance"] > 0,
    "feature",
].tolist()

if not selected_features:
    selected_features = X_train_transformed.columns.tolist()

X_train_selected = X_train_transformed[selected_features].copy()
X_val_selected = X_val_transformed[selected_features].copy()
selected_categorical_features = [col for col in categorical_features if col in selected_features]

dropped_features = [col for col in X_train_transformed.columns if col not in selected_features]

wandb.log(
    {
        "feature_selection/input_feature_count": X_train_transformed.shape[1],
        "feature_selection/selected_feature_count": len(selected_features),
        "feature_selection/dropped_feature_count": len(dropped_features),
        "feature_selection/selected_ratio": len(selected_features) / X_train_transformed.shape[1],
        "feature_selection/importance": wandb.Table(dataframe=feature_selection_importance),
        "feature_selection/selected_features": wandb.Table(
            dataframe=pd.DataFrame({"feature": selected_features})
        ),
        "feature_selection/dropped_features": wandb.Table(
            dataframe=pd.DataFrame({"feature": dropped_features})
        ),
    }
)
feature_selection_run.summary["selected_feature_count"] = len(selected_features)
feature_selection_run.summary["dropped_feature_count"] = len(dropped_features)
feature_selection_run.finish()

print(f"Selected {len(selected_features)} of {X_train_transformed.shape[1]} features for Optuna training.")

iteration,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇██
train_l1,█▇▆▅▅▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
validation_l1,█▇▇▆▆▅▄▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
iteration,99


[25]	train's l1: 3247.91	validation's l1: 2870.02
[50]	train's l1: 2014.16	validation's l1: 1797.13
[75]	train's l1: 1855.21	validation's l1: 1707.94
[100]	train's l1: 1794.65	validation's l1: 1687.44


feature_selection/dropped_feature_count,▁
feature_selection/input_feature_count,▁
feature_selection/selected_feature_count,▁
feature_selection/selected_ratio,▁
iteration,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▇▇▇▇▇▇▇▇█
train_l1,█▇▇▆▅▅▄▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
validation_l1,█▇▆▆▅▄▄▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_iteration,0
dropped_feature_count,35
feature_selection/dropped_feature_count,35
feature_selection/input_feature_count,82


Selected 47 of 82 features for Optuna training.


In [ ]:
def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 256),
        "max_depth": trial.suggest_int("max_depth", 5, 20),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 0.1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 0.1),
    }

    model_params = {**base_params, **params}

    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        job_type="train",
        name=f"lightgbm-optuna-trial-{trial.number}",
        tags=["lightgbm", "optuna", "tpe", "time-split"],
        config={
            **split_summary,
            "holiday_weight": HOLIDAY_WEIGHT,
            "feature_count": X_train_selected.shape[1],
            "categorical_features": selected_categorical_features,
            "feature_selection_rule": "feature_importance > 0",
            **model_params,
        },
        reinit=True,
    )

    print(f"\nTraining LightGBM model with Optuna trial {trial.number}: {params}")
    lgbm = lgb.LGBMRegressor(**model_params)

    lgbm.fit(
        X_train_selected,
        y_train,
        sample_weight=sample_weights_train,
        eval_set=[
            (X_train_selected, y_train),
            (X_val_selected, y_val),
        ],
        eval_names=["train", "validation"],
        eval_sample_weight=[sample_weights_train, sample_weights_val],
        eval_metric="mae",
        categorical_feature=selected_categorical_features,
        callbacks=[wandb_callback(), lgb.log_evaluation(period=25)],
    )
    log_summary(lgbm, save_model_checkpoint=False)

    y_pred_val = lgbm.predict(X_val_selected)
    weighted_mae = np.sum(np.abs(y_val - y_pred_val) * sample_weights_val) / np.sum(sample_weights_val)
    mae = np.mean(np.abs(y_val - y_pred_val))
    print(f"Validation Weighted MAE: {weighted_mae:.4f}")

    feature_importance = pd.DataFrame({
        "feature": X_train_selected.columns,
        "importance": lgbm.feature_importances_,
    }).sort_values("importance", ascending=False)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(y_val, y_pred_val, alpha=0.3)
    ax.plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2)
    ax.set_xlabel("Actual Weekly Sales")
    ax.set_ylabel("Predicted Weekly Sales")
    ax.set_title(f"Trial {trial.number}: Actual vs. Predicted Weekly Sales")
    wandb.log({
        "validation/weighted_mae": weighted_mae,
        "validation/mae": mae,
        "model/feature_importance": wandb.Table(dataframe=feature_importance.head(50)),
        "plots/actual_vs_predicted": wandb.Image(fig),
    })
    plt.close(fig)

    run.summary["best_validation_weighted_mae"] = weighted_mae
    run.finish()

    return weighted_mae

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("\n--- Optuna Hyperparameter Tuning Results ---")
print(f"Number of finished trials: {len(study.trials)}")
print(f"Best trial:")

trial = study.best_trial
print(f"  Value: {trial.value:.4f}")
print(f"  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

best_model_params = {**base_params, **study.best_params}
best_model = lgb.LGBMRegressor(**best_model_params)

best_model.fit(
    X_train_selected,
    y_train,
    sample_weight=sample_weights_train,
    eval_set=[
        (X_train_selected, y_train),
        (X_val_selected, y_val),
    ],
    eval_names=["train", "validation"],
    eval_sample_weight=[sample_weights_train, sample_weights_val],
    eval_metric="mae",
    categorical_feature=selected_categorical_features,
    callbacks=[lgb.log_evaluation(period=25)],
)

print("Optuna hyperparameter tuning complete.")


In [ ]:
# Register the already-fitted best LightGBM pipeline in W&B Model Registry.
# Run this cell after the Optuna/best_model cell has completed. This cell does not retrain the model.

import json
import cloudpickle
from pathlib import Path

required_objects = [
    "feature_pipeline",
    "selected_features",
    "selected_categorical_features",
    "best_model",
    "best_model_params",
    "study",
    "split_summary",
    "X_train_transformed",
    "X_train",
    "df_features",
    "df_stores",
]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise RuntimeError(
        "Run the previous cells first. Missing fitted objects: " + ", ".join(missing_objects)
    )


class LightGBMBestPipeline:
    """Fitted LightGBM pipeline bundle for W&B Model Registry."""

    def __init__(
        self,
        feature_pipeline,
        selected_features,
        selected_categorical_features,
        model,
        model_params,
        validation_weighted_mae,
        metadata,
        observed_history,
        external_features,
        stores,
    ):
        self.feature_pipeline = feature_pipeline
        self.selected_features = list(selected_features)
        self.selected_categorical_features = list(selected_categorical_features)
        self.model = model
        self.model_params = dict(model_params)
        self.validation_weighted_mae = float(validation_weighted_mae)
        self.metadata = dict(metadata)
        self.observed_history = observed_history.copy()
        self.external_features = external_features.copy()
        self.stores = stores.copy()
        self.markdown_cols = tuple(MARKDOWN_COLS)

    def _merge_raw(self, raw_df: pd.DataFrame) -> pd.DataFrame:
        frame = raw_df.copy()
        frame["__input_order"] = np.arange(len(frame))
        frame["Date"] = pd.to_datetime(frame["Date"])
        needs_external = not {"Type", "Size"}.issubset(frame.columns) or not set(self.markdown_cols).issubset(frame.columns)
        if needs_external:
            external = self.external_features.copy()
            external["Date"] = pd.to_datetime(external["Date"])
            external = external.drop(columns="IsHoliday", errors="ignore")
            frame = frame.merge(external, on=["Store", "Date"], how="left", validate="many_to_one")
            frame = frame.merge(self.stores, on="Store", how="left", validate="many_to_one")
        if "Weekly_Sales" not in frame.columns:
            frame["Weekly_Sales"] = np.nan
        return frame.sort_values("__input_order").drop(columns="__input_order").reset_index(drop=True)

    def transform_features(self, raw_df: pd.DataFrame) -> pd.DataFrame:
        prepared = self._merge_raw(raw_df)
        prepared = add_safe_lag_52(prepared, observed_history=self.observed_history)
        transformed = self.feature_pipeline.transform(prepared)
        missing = [col for col in self.selected_features if col not in transformed.columns]
        if missing:
            raise ValueError(f"Missing selected features after preprocessing: {missing}")
        return transformed[self.selected_features].copy()

    def predict(self, raw_df: pd.DataFrame) -> np.ndarray:
        features = self.transform_features(raw_df)
        predictions = self.model.predict(features)
        return np.clip(predictions, 0, None)


best_validation_weighted_mae = float(study.best_value)
registry_output_dir = Path("artifacts/lightgbm_registry")
registry_output_dir.mkdir(parents=True, exist_ok=True)

pipeline_bundle = LightGBMBestPipeline(
    feature_pipeline=feature_pipeline,
    selected_features=selected_features,
    selected_categorical_features=selected_categorical_features,
    model=best_model,
    model_params=best_model_params,
    validation_weighted_mae=best_validation_weighted_mae,
    metadata={
        **split_summary,
        "model_family": "LightGBM",
        "feature_selection_rule": "feature_importance > 0",
        "input_feature_count": int(X_train_transformed.shape[1]),
        "selected_feature_count": int(len(selected_features)),
        "categorical_feature_count": int(len(selected_categorical_features)),
        "safe_lag_features": ["SalesLag52", "SalesLag52_available"],
        "lag52_missing_handling": "left_as_nan_for_lightgbm_native_missing_value_handling",
        "raw_input_contract": "accepts raw test.csv and merges stored features/stores",
        "removed_unsafe_features": ["lag_1", "lag_4", "lag_13", "rolling_mean_4", "rolling_std_4", "rolling_mean_13", "rolling_std_13"],
    },
    observed_history=X_train,
    external_features=df_features,
    stores=df_stores,
)

pipeline_path = registry_output_dir / "lightgbm_best_pipeline.pkl"
with pipeline_path.open("wb") as file:
    cloudpickle.dump(pipeline_bundle, file)

selected_features_path = registry_output_dir / "selected_features.json"
selected_features_path.write_text(json.dumps(list(selected_features), indent=2))

model_params_path = registry_output_dir / "best_model_params.json"
model_params_path.write_text(json.dumps(best_model_params, indent=2, default=str))

registry_run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    job_type="model_registration",
    name="LightGBM_Best_Model_Registry",
    tags=["lightgbm", "best-model", "pipeline", "model-registry"],
    config={
        **split_summary,
        "best_validation_weighted_mae": best_validation_weighted_mae,
        "feature_selection_rule": "feature_importance > 0",
        "selected_feature_count": len(selected_features),
        "selected_categorical_features": selected_categorical_features,
        **best_model_params,
    },
    reinit=True,
)

model_artifact = wandb.Artifact(
    name="walmart-lightgbm-best-pipeline",
    type="model",
    description="Best LightGBM model packaged with the fitted feature-engineering pipeline and selected feature list.",
    metadata={
        "model_family": "LightGBM",
        "best_validation_weighted_mae": best_validation_weighted_mae,
        "feature_selection_rule": "feature_importance > 0",
        "selected_feature_count": len(selected_features),
        "categorical_feature_count": len(selected_categorical_features),
        "registry_target": "wandb-registry-model/Walmart_LightGBM_Pipeline",
        "raw_input_contract": "accepts raw test.csv and merges stored features/stores",
    },
)
model_artifact.add_file(str(pipeline_path))
model_artifact.add_file(str(selected_features_path))
model_artifact.add_file(str(model_params_path))

logged_artifact = registry_run.log_artifact(model_artifact, aliases=["best", "latest"])
registry_run.link_artifact(
    logged_artifact,
    target_path="wandb-registry-model/Walmart_LightGBM_Pipeline",
    aliases=["best-lightgbm", "latest", "champion"],
)

registry_run.summary["best_validation_weighted_mae"] = best_validation_weighted_mae
registry_run.summary["registry_target"] = "wandb-registry-model/Walmart_LightGBM_Pipeline"
registry_run.summary["pipeline_artifact"] = "walmart-lightgbm-best-pipeline"
registry_run.finish()

print("Registered best LightGBM pipeline in W&B Model Registry:")
print("  artifact: walmart-lightgbm-best-pipeline")
print("  registry: wandb-registry-model/Walmart_LightGBM_Pipeline")
print(f"  validation weighted MAE: {best_validation_weighted_mae:.4f}")



